# Step 9 — Verification Supervisor
**Spec trace:** REQ-200-VS1, REQ-700-090/091/092, REQ-700-VS1, REQ-200-170

The VS is the **final structural gate** before response delivery.  
It checks pipeline integrity — not response content (that is the Evaluator's job).

| Check | Suspended in Crisis Mode? |
|---|---|
| C1 Evaluator gate | No |
| C2 Scope routing integrity | No |
| C3 Mode-distress alignment | No |
| C4 Crisis resources present | No |
| C5 Evidence pipeline (Guidance) | **Yes** |
| C6 Agent contamination | **Yes** |
| C7 Loop limit (regen < 2) | No |


## Setup

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from agents.verification_supervisor import VerificationSupervisorAgent, SAFE_FALLBACK_RESPONSE
from docs.schemas.acp_schemas import (
    DistressLevel, EvaluationPayload, EvaluationVerdict,
    OperationalMode, ResponseContextPayload, ScopeClassifierDecision,
    ScopeDecision, VerificationResult,
    SignalPayload, StrategyPayload, SynthesizedEvidence,
    EvidenceItem, EvidenceTier, SourceTier, CrisisResource,
)
from datetime import datetime, timezone
from uuid import uuid4

vs = VerificationSupervisorAgent()
print('VS agent instantiated OK')
print(f'SAFE_FALLBACK_RESPONSE[:60]: {SAFE_FALLBACK_RESPONSE[:60]}...')

VS agent instantiated OK
SAFE_FALLBACK_RESPONSE[:60]: I'm here to support you. If you're going through something d...


## Helper factories
Minimally valid objects — only the fields under test vary per section.

In [2]:
from datetime import datetime, timezone
_NOW = datetime.now(tz=timezone.utc)

def make_signal(distress=DistressLevel.LOW):
    return SignalPayload(
        distress_level=distress, confidence=0.90,
        emotional_states=[], cognitive_patterns=[],
        behavioral_indicators=[], risk_indicators=[], support_needs=[],
    )

def make_strategy(mode=OperationalMode.COMFORT, distress=DistressLevel.LOW):
    return StrategyPayload(
        mode=mode, distress_level=distress,
        tone_guidance='warm and validating',
        framing_strategy='validate and support',
    )

def make_context(
    mode=OperationalMode.COMFORT, distress=DistressLevel.LOW,
    evidence=None, crisis_resources=None,
):
    return ResponseContextPayload(
        mode=mode, signals=make_signal(distress),
        strategy=make_strategy(mode, distress),
        synthesized_evidence=evidence, crisis_resources=crisis_resources,
    )

def make_eval(verdict=EvaluationVerdict.PASS):
    return EvaluationPayload(
        verdict=verdict, safety_check=(verdict == EvaluationVerdict.PASS),
        tone_check=True, hallucination_check=True,
    )

def make_scope(decision=ScopeDecision.IN_SCOPE, confidence=0.95):
    redirect = 'I can only help with emotional wellbeing.' if decision == ScopeDecision.OUT_OF_SCOPE else None
    return ScopeClassifierDecision(decision=decision, confidence=confidence, warm_redirect=redirect)

def make_evidence():
    return SynthesizedEvidence(
        summary='1 source retrieved.', confidence=0.90,
        grey_literature_used=False, source_disagreement=False,
        citations=[EvidenceItem(
            title='CBT: Evidence Review',
            abstract='CBT has strong evidence. ' * 4,
            source_name='PubMed', url='https://pubmed.ncbi.nlm.nih.gov/1',
            evidence_tier=EvidenceTier.PEER_REVIEWED, source_tier=SourceTier.PRIMARY,
            retrieved_at=_NOW,
        )],
    )

resource = CrisisResource(name='Lifeline', number='13 11 14', tier='baseline')
print('Helpers defined.')

Helpers defined.


## 1 — Happy path: all checks pass (COMFORT mode)

In [3]:
ctx   = make_context(mode=OperationalMode.COMFORT, distress=DistressLevel.LOW)
eval_ = make_eval(EvaluationVerdict.PASS)
scope = make_scope(ScopeDecision.IN_SCOPE)

result = vs.verify(ctx, eval_, scope, regen_count=0)
print(f'passed                  : {result.passed}')
print(f'safe_fallback_triggered : {result.safe_fallback_triggered}')
print(f'crisis_mode_active      : {result.crisis_mode_active}')
print(f'failure_reasons         : {result.failure_reasons}')
assert result.passed
assert not result.safe_fallback_triggered
print()
print('OK - happy path passed as expected')

passed                  : True
safe_fallback_triggered : False
crisis_mode_active      : False
failure_reasons         : []

OK - happy path passed as expected


## 2 — C1: Evaluator FAIL reaching VS (orchestrator bug)

In [4]:
result = vs.verify(make_context(), make_eval(EvaluationVerdict.FAIL), make_scope())
print(f'passed          : {result.passed}')
print(f'failure_reasons : {result.failure_reasons}')
assert not result.passed
assert result.safe_fallback_triggered
assert any('C1' in r for r in result.failure_reasons)
print()
print('OK - C1 correctly caught evaluator non-PASS')

VS: 1 check(s) failed — safe fallback triggered. Reasons: ["[C1/REQ-700-090] Evaluator verdict is 'fail' — pipeline must not reach VS with a non-PASS verdict. Rejection reasons: []"]


passed          : False
failure_reasons : ["[C1/REQ-700-090] Evaluator verdict is 'fail' — pipeline must not reach VS with a non-PASS verdict. Rejection reasons: []"]

OK - C1 correctly caught evaluator non-PASS


## 3 — C2: OUT_OF_SCOPE input reached the VS (routing failure)

In [5]:
result = vs.verify(make_context(), make_eval(), make_scope(ScopeDecision.OUT_OF_SCOPE, 0.95))
print(f'passed          : {result.passed}')
print(f'failure_reasons : {result.failure_reasons}')
assert not result.passed
assert any('C2' in r for r in result.failure_reasons)
print()
print('OK - C2 correctly caught OUT_OF_SCOPE routing failure')

VS: 1 check(s) failed — safe fallback triggered. Reasons: ["[C2/REQ-700-091] Scope classifier returned OUT_OF_SCOPE (reason: 'I can only help with emotional wellbeing.') — request should have been deflected at STEP 0 and must not reach the VS."]


passed          : False
failure_reasons : ["[C2/REQ-700-091] Scope classifier returned OUT_OF_SCOPE (reason: 'I can only help with emotional wellbeing.') — request should have been deflected at STEP 0 and must not reach the VS."]

OK - C2 correctly caught OUT_OF_SCOPE routing failure


## 4 — C3: Mode-distress misalignment
CRITICAL distress must route to CRISIS. ELEVATED must not go to COMFORT.

In [6]:
cases = [
    ('CRITICAL->COMFORT', OperationalMode.COMFORT,  DistressLevel.CRISIS),
    ('ELEVATED->COMFORT', OperationalMode.COMFORT,  DistressLevel.HIGH),
    ('NONE->CRISIS',      OperationalMode.CRISIS,   DistressLevel.LOW),
]
for label, mode, distress in cases:
    ctx = make_context(mode=mode, distress=distress)
    r   = vs.verify(ctx, make_eval(), make_scope())
    c3  = [x for x in r.failure_reasons if 'C3' in x]
    assert not r.passed and c3, f'Expected C3 failure for {label}'
    print(f'[{label}] passed={r.passed}  reason: {c3[0][:80]}')
print()
print('OK - C3 caught all three misalignment cases')

VS: 2 check(s) failed — safe fallback triggered. Reasons: ["[C3/REQ-200-VS1] CRITICAL distress but mode='comfort' — CRITICAL distress MUST route to CRISIS mode.", '[C4/SPEC-300§5] CRITICAL distress detected but crisis_resources is absent or empty — Crisis Mode MUST populate crisis resources.']


VS: 1 check(s) failed — safe fallback triggered. Reasons: ["[C3/REQ-200-VS1] ELEVATED distress but mode='comfort' — ELEVATED distress MUST route to GUIDANCE or CRISIS mode."]


VS: 1 check(s) failed — safe fallback triggered. Reasons: ["[C3/REQ-200-VS1] NONE distress but mode='crisis' — Crisis resources delivered to a non-distress input is a routing anomaly."]


[CRITICAL->COMFORT] passed=False  reason: [C3/REQ-200-VS1] CRITICAL distress but mode='comfort' — CRITICAL distress MUST r
[ELEVATED->COMFORT] passed=False  reason: [C3/REQ-200-VS1] ELEVATED distress but mode='comfort' — ELEVATED distress MUST r
[NONE->CRISIS] passed=False  reason: [C3/REQ-200-VS1] NONE distress but mode='crisis' — Crisis resources delivered to

OK - C3 caught all three misalignment cases


## 5 — C4: CRITICAL distress with no crisis_resources

In [7]:
resource = CrisisResource(name='Lifeline', number='13 11 14', tier='baseline')

# C4 fail: CRITICAL with no resources
ctx  = make_context(mode=OperationalMode.CRISIS, distress=DistressLevel.CRISIS, crisis_resources=None)
r_no = vs.verify(ctx, make_eval(), make_scope())
assert any('C4' in x for x in r_no.failure_reasons)
print(f'[no resources]    passed={r_no.passed}  C4 triggered: True')

# C4 pass: CRITICAL with resources
ctx2  = make_context(mode=OperationalMode.CRISIS, distress=DistressLevel.CRISIS, crisis_resources=[resource])
r_yes = vs.verify(ctx2, make_eval(), make_scope())
c4    = [x for x in r_yes.failure_reasons if 'C4' in x]
assert not c4
print(f'[with resources]  C4 clean: True')
print()
print('OK - C4 correctly enforces crisis_resources presence')

VS: 1 check(s) failed — safe fallback triggered. Reasons: ['[C4/SPEC-300§5] CRITICAL distress detected but crisis_resources is absent or empty — Crisis Mode MUST populate crisis resources.']


[no resources]    passed=False  C4 triggered: True
[with resources]  C4 clean: True

OK - C4 correctly enforces crisis_resources presence


## 6 — C5 + C6: Evidence integrity and contamination
C5: GUIDANCE mode must have `synthesized_evidence`.  
C6: COMFORT must not carry evidence; GUIDANCE must not carry crisis_resources.

In [8]:
resource = CrisisResource(name='Lifeline', number='13 11 14', tier='baseline')

# C5 fail: GUIDANCE with no evidence
ctx  = make_context(mode=OperationalMode.GUIDANCE, distress=DistressLevel.HIGH, evidence=None)
r    = vs.verify(ctx, make_eval(), make_scope())
assert any('C5' in x for x in r.failure_reasons)
print(f'[C5 GUIDANCE/no-evidence]       C5 triggered: True')

# C5 pass: GUIDANCE with evidence
ctx  = make_context(mode=OperationalMode.GUIDANCE, distress=DistressLevel.HIGH, evidence=make_evidence())
r    = vs.verify(ctx, make_eval(), make_scope())
assert not [x for x in r.failure_reasons if 'C5' in x]
print(f'[C5 GUIDANCE/evidence-present]  C5 clean: True')

# C6 fail: COMFORT with evidence
ctx  = make_context(mode=OperationalMode.COMFORT, distress=DistressLevel.LOW, evidence=make_evidence())
r    = vs.verify(ctx, make_eval(), make_scope())
assert any('C6' in x for x in r.failure_reasons)
print(f'[C6 COMFORT/evidence]           C6 triggered: True')

# C6 fail: GUIDANCE with crisis resources
ctx  = make_context(mode=OperationalMode.GUIDANCE, distress=DistressLevel.HIGH,
                    evidence=make_evidence(), crisis_resources=[resource])
r    = vs.verify(ctx, make_eval(), make_scope())
assert any('C6' in x for x in r.failure_reasons)
print(f'[C6 GUIDANCE/crisis_resources]  C6 triggered: True')
print()
print('OK - C5 and C6 operating correctly')

VS: 1 check(s) failed — safe fallback triggered. Reasons: ['[C5/REQ-700-091] GUIDANCE mode requires synthesized_evidence but it is None — evidence retrieval or synthesis step was skipped.']


VS: 1 check(s) failed — safe fallback triggered. Reasons: ['[C6/REQ-700-091] COMFORT mode must not carry synthesized_evidence — evidence pipeline should not have run in this mode.']


VS: 1 check(s) failed — safe fallback triggered. Reasons: ['[C6/REQ-700-091] GUIDANCE mode must not carry crisis_resources — crisis pathway was not activated for this request.']


[C5 GUIDANCE/no-evidence]       C5 triggered: True
[C5 GUIDANCE/evidence-present]  C5 clean: True
[C6 COMFORT/evidence]           C6 triggered: True
[C6 GUIDANCE/crisis_resources]  C6 triggered: True

OK - C5 and C6 operating correctly


## 7 — C7: Regeneration loop limit (REQ-200-170)
Pipeline MUST NOT iterate more than 2 times. VS blocks the 3rd attempt.

In [9]:
ctx = make_context()
for regen in range(4):
    r       = vs.verify(ctx, make_eval(), make_scope(), regen_count=regen)
    blocked = any('C7' in x for x in r.failure_reasons)
    print(f'  regen_count={regen}  passed={r.passed}  C7_blocked={blocked}')

assert vs.verify(ctx, make_eval(), make_scope(), regen_count=0).passed
assert vs.verify(ctx, make_eval(), make_scope(), regen_count=1).passed
assert not vs.verify(ctx, make_eval(), make_scope(), regen_count=2).passed
assert not vs.verify(ctx, make_eval(), make_scope(), regen_count=3).passed
print()
print('OK - C7 blocks at regen_count >= 2 per REQ-200-170')

VS: 1 check(s) failed — safe fallback triggered. Reasons: ['[C7/REQ-200-170] regen_count=2 has reached the maximum of 2. Further regeneration is blocked. Safe fallback response will be emitted.']


VS: 1 check(s) failed — safe fallback triggered. Reasons: ['[C7/REQ-200-170] regen_count=3 has reached the maximum of 2. Further regeneration is blocked. Safe fallback response will be emitted.']


VS: 1 check(s) failed — safe fallback triggered. Reasons: ['[C7/REQ-200-170] regen_count=2 has reached the maximum of 2. Further regeneration is blocked. Safe fallback response will be emitted.']


VS: 1 check(s) failed — safe fallback triggered. Reasons: ['[C7/REQ-200-170] regen_count=3 has reached the maximum of 2. Further regeneration is blocked. Safe fallback response will be emitted.']


  regen_count=0  passed=True  C7_blocked=False
  regen_count=1  passed=True  C7_blocked=False
  regen_count=2  passed=False  C7_blocked=True
  regen_count=3  passed=False  C7_blocked=True

OK - C7 blocks at regen_count >= 2 per REQ-200-170


## 8 — Crisis Mode: C5 and C6 suspended (REQ-700-VS1)
When `distress=CRITICAL`, evidence-pipeline and contamination checks are suspended.  
Speed and safety-resource delivery take priority over structural purity.

In [10]:
resource = CrisisResource(name='Lifeline', number='13 11 14', tier='baseline')

ctx = make_context(
    mode=OperationalMode.CRISIS,
    distress=DistressLevel.CRISIS,
    evidence=None,            # would fail C5 in normal mode
    crisis_resources=[resource],
)
result = vs.verify(ctx, make_eval(), make_scope())
print(f'crisis_mode_active      : {result.crisis_mode_active}')
c5 = [r for r in result.failure_reasons if 'C5' in r]
c6 = [r for r in result.failure_reasons if 'C6' in r]
print(f'C5 failures (expect 0)  : {c5}')
print(f'C6 failures (expect 0)  : {c6}')
assert result.crisis_mode_active
assert not c5, 'C5 must be suspended in Crisis Mode'
assert not c6, 'C6 must be suspended in Crisis Mode'
print()
print('OK - Crisis Mode correctly suspends C5 and C6 per REQ-700-VS1')

crisis_mode_active      : True
C5 failures (expect 0)  : []
C6 failures (expect 0)  : []

OK - Crisis Mode correctly suspends C5 and C6 per REQ-700-VS1


## 9 — Safe fallback response (REQ-700-092)
On any VS failure, `safe_fallback_triggered=True` and the pipeline emits `SAFE_FALLBACK_RESPONSE`.

In [11]:
ctx    = make_context(mode=OperationalMode.GUIDANCE, distress=DistressLevel.HIGH, evidence=None)
result = vs.verify(ctx, make_eval(), make_scope())
print(f'passed                  : {result.passed}')
print(f'safe_fallback_triggered : {result.safe_fallback_triggered}')
print(f'failure_reasons         : {result.failure_reasons}')
print()
print('Pipeline would deliver:')
print(f'  "{SAFE_FALLBACK_RESPONSE}"')
assert not result.passed
assert result.safe_fallback_triggered
print()
print('OK - safe_fallback_triggered correctly set on VS failure')

VS: 1 check(s) failed — safe fallback triggered. Reasons: ['[C5/REQ-700-091] GUIDANCE mode requires synthesized_evidence but it is None — evidence retrieval or synthesis step was skipped.']


passed                  : False
safe_fallback_triggered : True
failure_reasons         : ['[C5/REQ-700-091] GUIDANCE mode requires synthesized_evidence but it is None — evidence retrieval or synthesis step was skipped.']

Pipeline would deliver:
  "I'm here to support you. If you're going through something difficult right now, please reach out to a trusted person or a professional who can help. You don't have to face this alone."

OK - safe_fallback_triggered correctly set on VS failure


## 10 — Sandbox
Edit the cell below to test your own pipeline state combinations.

In [12]:
# --- Edit these to test custom scenarios ---
my_ctx = make_context(
    mode=OperationalMode.GUIDANCE,
    distress=DistressLevel.HIGH,
    evidence=make_evidence(),   # swap to None to trigger C5
)
my_eval  = make_eval(EvaluationVerdict.PASS)
my_scope = make_scope(ScopeDecision.IN_SCOPE, confidence=0.88)

result = vs.verify(my_ctx, my_eval, my_scope, regen_count=0)
print(f'passed                  : {result.passed}')
print(f'crisis_mode_active      : {result.crisis_mode_active}')
print(f'safe_fallback_triggered : {result.safe_fallback_triggered}')
print(f'failure_reasons         : {result.failure_reasons}')

passed                  : True
crisis_mode_active      : False
safe_fallback_triggered : False
failure_reasons         : []
